# CIFAR-10 Classifier — Analysis

Training now happens in `train.py`:

    uv run python projects/cifar10/classify/train.py

This notebook only loads the resulting checkpoint and metrics for analysis:
training curves, confusion matrix, per-class report, and sample predictions.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from chimera.data import CIFAR10DataModule
from chimera.models import ResNet

RUN_DIR = "/mnt/ai/runs/cifar10/classify"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 10
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

## Load checkpoint

In [ ]:
ckpt = torch.load(
    f"{RUN_DIR}/checkpoints/classifier.ckpt", map_location="cpu", weights_only=False
)
model_state = {
    k.removeprefix("model."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = ResNet(in_channels=3, num_classes=NUM_CLASSES)
model.load_state_dict(model_state)
model.to(DEVICE).eval()
print(
    f"loaded classifier ({sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt['epoch']})"
)

## Data

Load the CIFAR-10 test split and preview a few labelled samples.

In [ ]:
dm = CIFAR10DataModule(pin_memory=False, num_workers=0, data_dir=DATA_DIR)
dm.prepare_data()
dm.setup("test")
test_loader = dm.test_dataloader()

images, labels = next(iter(test_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("CIFAR-10 test samples")
for ax, img, label in zip(axes.flat, images, labels):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(classes[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Training curves

Reads the CSV log written alongside the wandb run (`chimera.utils.build_run_loggers`).

In [ ]:
csv_path = sorted(glob.glob(f"{RUN_DIR}/csv/version_*/metrics.csv"))[-1]
metrics = pd.read_csv(csv_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")
for ax, key, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    for stage in ["train", "val"]:
        col = f"{stage}/{key}"
        if col not in metrics.columns:
            continue
        d = metrics.dropna(subset=[col])
        ax.plot(d["step"], d[col], marker="o" if stage == "val" else None, label=stage)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

## Confusion matrix + per-class report

(Also logged live during training as a wandb image + table — see `ClassifierModule.log_confusion_matrix`.)

In [ ]:
all_preds, all_targets = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds = model(x.to(DEVICE)).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_targets.append(y)
all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
for t, p in zip(all_targets, all_preds):
    confusion[t, p] += 1

accuracy = confusion.diag().sum().item() / confusion.sum().item()
print(f"Test accuracy: {accuracy:.4f}\n")

support = confusion.sum(dim=1)
tp = confusion.diag()
precision = tp / confusion.sum(dim=0).clamp_min(1)
recall = tp / support.clamp_min(1)
f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-9)
print(f"{'class':>12}  {'precision':>9}  {'recall':>9}  {'f1':>9}  {'support':>7}")
for c in range(NUM_CLASSES):
    print(
        f"{classes[c]:>12}  {precision[c]:>9.3f}  {recall[c]:>9.3f}  {f1[c]:>9.3f}  {support[c]:>7d}"
    )

confusion_np = confusion.numpy()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(confusion_np, cmap="Blues")
fig.colorbar(im, ax=ax)
ax.set_title("Test Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticklabels(classes)
threshold = confusion_np.max() / 2
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j,
            i,
            confusion_np[i, j],
            ha="center",
            va="center",
            color="white" if confusion_np[i, j] > threshold else "black",
        )
plt.tight_layout()
plt.show()

## Sample predictions

Visualize predictions on a batch of test images; titles show `pred (true)` and
are coloured green when correct, red when wrong.

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(DEVICE)).argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(15, 8))
fig.suptitle("Sample predictions — pred (true)")
for ax, img, pred, label in zip(axes.flat, images, preds, labels):
    ax.imshow(img.permute(1, 2, 0).numpy())
    correct = pred.item() == label.item()
    ax.set_title(
        f"{classes[pred]} ({classes[label]})",
        fontsize=8,
        color="green" if correct else "red",
    )
    ax.axis("off")
plt.tight_layout()
plt.show()